# SafeGuard K3 Inspector – Training Ulang YOLOv8s & Audit Akurasi APD
### Klasifikasi Kepatuhan Penggunaan APD Pekerja Menggunakan YOLOv8

Notebook ini dirancang untuk melatih ulang model deteksi objek APD K3 berbasis **YOLOv8s** di Kaggle GPU, dengan fokus audit dan peningkatan akurasi kelas **Person (Pekerja)**.

---
### 📋 Konfigurasi Training:
* **Base Model**: `yolov8s.pt` (Small ~22 MB, kapasitas ekstraksi representasi spasial lebih kuat dari nano)
* **Image Size (`imgsz`)**: `640`
* **Epochs**: `80`
* **Patience**: `15` (Early stopping jika tidak ada perbaikan mAP50-95 selama 15 epoch)
* **Batch Size**: `4` (Optimal dan stabil untuk GPU memory Kaggle T4/P100)
* **Cache**: `False` (Mencegah out-of-RAM pada environment Kaggle)
* **Plots**: `True` (Menyimpan visualisasi metrik evaluasi)
* **Save**: `True` (Menyimpan checkpoint `best.pt` dan `last.pt`)

---
### 🎯 6 Kelas Deteksi K3:
`0: Gloves`, `1: Hard_hat`, `2: Mask`, `3: Person`, `4: Safety_boots`, `5: Vest`


In [ ]:
# 1. Instalasi dependensi Ultralytics YOLOv8 dan visualisasi
!pip install -q ultralytics pyyaml matplotlib seaborn pandas opencv-python

import os
import glob
import yaml
import cv2
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
else:
    print("PERINGATAN: GPU tidak terdeteksi. Aktifkan GPU T4 di menu Settings Kaggle!")


## 2. Deteksi & Penyiapan Dataset
Cell ini secara otomatis mendeteksi apakah dataset diunggah sebagai folder di `/kaggle/input/` atau sebagai file ZIP.


In [ ]:
# 2. Cari file dataset di input Kaggle
DATASET_DIR = None

# Cek kemungkinan lokasi dataset di Kaggle Input
possible_paths = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
if possible_paths:
    DATASET_DIR = str(Path(possible_paths[0]).parent)
    print(f"✅ data.yaml ditemukan di: {DATASET_DIR}")
else:
    # Cek apakah ada file zip dataset
    zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zip_files:
        zip_path = zip_files[0]
        print(f"📦 Mengekstrak dataset dari {zip_path} ke /kaggle/working/dataset/...")
        DATASET_DIR = "/kaggle/working/dataset"
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATASET_DIR)
        print("✅ Ekstraksi selesai!")
    else:
        # Fallback direktori lokal
        DATASET_DIR = "/kaggle/working/dataset"
        print("⚠️ Pastikan dataset diunggah ke Kaggle Input. Jalankan ekstraksi manual jika diperlukan.")

yaml_file = os.path.join(DATASET_DIR, "data.yaml")
if os.path.exists(yaml_file):
    with open(yaml_file, 'r') as f:
        data_cfg = yaml.safe_load(f)
    print(f"Isi data.yaml:")
    print(yaml.dump(data_cfg, default_flow_style=False))
else:
    print(f"❌ File data.yaml tidak ditemukan di {DATASET_DIR}")


## 3. Audit Distribusi Anotasi & Deteksi Ground-Truth Inkonsisten
Pada tahap ini, kita menghitung frekuensi bounding box setiap kelas dan mendeteksi sampel foto yang memiliki Helm/Rompi tetapi **anotasi Person-nya tidak ada (hilang)**.


In [ ]:
# 3. Audit distribusi kelas pada seluruh split dataset
CLASSES = ['Gloves', 'Hard_hat', 'Mask', 'Person', 'Safety_boots', 'Vest']
splits = ['train', 'valid', 'test']

distribution = {s: Counter() for s in splits}
file_counts = {s: 0 for s in splits}
missing_person_samples = []

for s in splits:
    # Cari direktori label
    label_dirs = [
        os.path.join(DATASET_DIR, s, "labels"),
        os.path.join(DATASET_DIR, f"../{s}/labels"),
        os.path.join(DATASET_DIR, s)
    ]
    lbl_dir = None
    for d in label_dirs:
        if os.path.exists(d) and glob.glob(os.path.join(d, "*.txt")):
            lbl_dir = d
            break
            
    if lbl_dir:
        txt_files = glob.glob(os.path.join(lbl_dir, "*.txt"))
        file_counts[s] = len(txt_files)
        
        for txt in txt_files:
            with open(txt, 'r') as f:
                lines = [l.strip().split() for l in f.readlines() if l.strip()]
            
            classes_in_file = set()
            for line in lines:
                try:
                    c_id = int(line[0])
                    distribution[s][c_id] += 1
                    classes_in_file.add(c_id)
                except ValueError:
                    pass
                    
            # Deteksi ketiadaan Person pada gambar ber-APD
            has_apd = bool(classes_in_file.intersection({1, 5})) # 1=Hard_hat, 5=Vest
            if has_apd and 3 not in classes_in_file: # 3=Person
                missing_person_samples.append((s, os.path.basename(txt)))

# Buat DataFrame distribusi
df_data = []
for i, c_name in enumerate(CLASSES):
    tr = distribution['train'][i]
    val = distribution['valid'][i]
    te = distribution['test'][i]
    tot = tr + val + te
    df_data.append({
        "ID": i,
        "Class Name": c_name,
        "Train": tr,
        "Valid": val,
        "Test": te,
        "Total Box": tot
    })

df_dist = pd.DataFrame(df_data)
print("=" * 65)
print("TABEL DISTRIBUSI ANOTASI DATASET K3:")
print("=" * 65)
print(df_dist.to_string(index=False))
print("-" * 65)
print(f"Total File Gambar per Split: {file_counts}")
print(f"Jumlah Gambar dengan APD (Helm/Rompi) tetapi TANPA label Person: {len(missing_person_samples)}")
if len(missing_person_samples) > 0:
    print(f"⚠️ CONTOH SAMPEL ANOTASI INKONSISTEN (Person hilang): {missing_person_samples[:5]}")


In [ ]:
# 4. Visualisasi Grafik Batang Distribusi Kelas
plt.figure(figsize=(10, 5))
bar_width = 0.25
x = np.arange(len(CLASSES))

plt.bar(x - bar_width, df_dist['Train'], width=bar_width, label='Train', color='#17324D')
plt.bar(x, df_dist['Valid'], width=bar_width, label='Valid', color='#E9A826')
plt.bar(x + bar_width, df_dist['Test'], width=bar_width, label='Test', color='#198754')

plt.xlabel('Kelas Objek APD', fontweight='bold')
plt.ylabel('Jumlah Bounding Box', fontweight='bold')
plt.title('Distribusi Bounding Box per Kelas pada Dataset SafeGuard K3', fontsize=12, fontweight='bold')
plt.xticks(x, CLASSES, rotation=15)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150)
plt.show()


## 4. Verifikasi Visual Bounding Box
Menampilkan sampel gambar proyek beserta bounding box anotasi asli untuk memastikan ketepatan koordinat spasial.


In [ ]:
# 5. Fungsi visualisasi gambar dengan bounding box
CLASS_COLORS = {
    0: (13, 148, 136),   # Gloves: Teal
    1: (25, 135, 84),    # Hard_hat: Hijau
    2: (2, 132, 199),    # Mask: Sky Blue
    3: (23, 50, 77),     # Person: Navy
    4: (124, 58, 237),   # Safety_boots: Ungu
    5: (233, 168, 38)    # Vest: Amber
}

def draw_gt_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                    
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    
                    color = CLASS_COLORS.get(cls_id, (100, 100, 100))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    label_text = CLASSES[cls_id] if cls_id < len(CLASSES) else str(cls_id)
                    cv2.putText(img, label_text, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

# Ambil 4 sampel gambar dari train
train_imgs = glob.glob(os.path.join(DATASET_DIR, "train", "images", "*.*"))
if not train_imgs:
    train_imgs = glob.glob(os.path.join(DATASET_DIR, "**", "*.jpg"), recursive=True)

sample_imgs = train_imgs[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, img_p in zip(axes.flatten(), sample_imgs):
    base = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(os.path.dirname(os.path.dirname(img_p)), "labels", base + ".txt")
    drawn = draw_gt_boxes(img_p, lbl_p)
    if drawn is not None:
        ax.imshow(drawn)
        ax.set_title(os.path.basename(img_p), fontsize=9)
    ax.axis('off')

plt.suptitle("Contoh Anotasi Ground-Truth Dataset SafeGuard K3", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Konfigurasi data_safeguard.yaml
Membuat file konfigurasi dataset lokal dengan path absolut yang valid untuk Ultralytics.


In [ ]:
# 6. Menulis data_safeguard.yaml lokal
local_yaml_path = "/kaggle/working/data_safeguard.yaml"

train_path = os.path.abspath(os.path.join(DATASET_DIR, "train", "images"))
val_path = os.path.abspath(os.path.join(DATASET_DIR, "valid", "images"))
test_path = os.path.abspath(os.path.join(DATASET_DIR, "test", "images"))

yaml_data = {
    'path': os.path.abspath(DATASET_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 6,
    'names': CLASSES
}

with open(local_yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("✅ File data_safeguard.yaml berhasil dibuat:")
with open(local_yaml_path, 'r') as f:
    print(f.read())


## 6. Training YOLOv8s
Menjalankan training model `yolov8s.pt` dengan spesifikasi yang telah disetujui:
* **imgsz**: 640
* **epochs**: 80
* **patience**: 15
* **batch**: 4
* **cache**: False
* **plots**: True
* **save**: True


In [ ]:
from ultralytics import YOLO

# 7. Inisialisasi model pre-trained YOLOv8s
print("🚀 Memuat base model yolov8s.pt...")
model = YOLO('yolov8s.pt')

# Jalankan training
train_results = model.train(
    data=local_yaml_path,
    epochs=80,
    patience=15,
    batch=4,
    imgsz=640,
    cache=False,
    plots=True,
    save=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    workers=2,
    project='safeguardk3_training',
    name='yolov8s_k3_model',
    exist_ok=True
)

print("🎉 Training selesai!")


## 7. Evaluasi Model pada Split Test
Mengevaluasi performa `best.pt` pada set pengujian (`test`), dengan perhatian khusus pada metrik precision, recall, dan mAP untuk kelas **Person**.


In [ ]:
# 8. Evaluasi best.pt pada split test
best_weight = "/kaggle/working/safeguardk3_training/yolov8s_k3_model/weights/best.pt"

if os.path.exists(best_weight):
    print(f"✅ Memuat model terbaik: {best_weight}")
    trained_model = YOLO(best_weight)
    
    print("\n--- MENJALANKAN EVALUASI TEST SPLIT ---")
    test_metrics = trained_model.val(
        data=local_yaml_path,
        split='test',
        imgsz=640,
        batch=4,
        conf=0.20,
        plots=True
    )
    
    print("\n" + "=" * 50)
    print("HASIL EVALUASI METRIK K3:")
    print("=" * 50)
    print(f"mAP50 (All Classes)    : {test_metrics.box.map50:.4f}")
    print(f"mAP50-95 (All Classes) : {test_metrics.box.map:.4f}")
    
    # Detail per kelas
    print("\nMetrik per Kelas (Presisi, Recall, mAP50):")
    for i, c_name in enumerate(CLASSES):
        p = test_metrics.box.p[i] if len(test_metrics.box.p) > i else 0.0
        r = test_metrics.box.r[i] if len(test_metrics.box.r) > i else 0.0
        map50 = test_metrics.box.map50s[i] if len(test_metrics.box.map50s) > i else 0.0
        highlight = " 🌟 (FOKUS AUDIT)" if c_name == "Person" else ""
        print(f"  {c_name:<15}: Precision={p:.3f} | Recall={r:.3f} | mAP50={map50:.3f}{highlight}")
else:
    print(f"❌ File {best_weight} tidak ditemukan.")


## 8. Uji Inferensi Komparatif pada Foto Proyek
Menguji ketahanan deteksi `Person` pada confidence 0.20 vs 0.25.


In [ ]:
# 9. Prediksi komparatif pada 3 gambar uji
test_images = glob.glob(os.path.join(DATASET_DIR, "test", "images", "*.*"))[:3]

if os.path.exists(best_weight) and test_images:
    for img_p in test_images:
        img_name = os.path.basename(img_p)
        
        res25 = trained_model(img_p, conf=0.25, verbose=False)[0]
        res20 = trained_model(img_p, conf=0.20, verbose=False)[0]
        
        c25 = [CLASSES[int(b.cls[0])] for b in res25.boxes]
        c20 = [f"{CLASSES[int(b.cls[0])]}:{float(b.conf[0]):.2f}" for b in res20.boxes]
        
        print(f"\nFoto: {img_name}")
        print(f"  Threshold 0.25 : {Counter(c25)}")
        print(f"  Threshold 0.20 : {Counter([c.split(':')[0] for c in c20])}")
        print(f"  Detail Conf    : {c20}")


## 9. Ekspor Model Akhir & Pembuatan File ZIP untuk Backend
Cell ini membungkus bobot `best.pt` dan grafik evaluasi ke dalam `safeguardk3_best_model.zip`.
File zip ini dapat langsung diunduh dari tab **Output** Kaggle dan disalin ke direktori proyek lokal:
`d:\Klasifikasi-APD-K3\artifacts\best.pt`.


In [ ]:
# 10. Paketkan model ke file zip agar mudah didownload
export_dir = "/kaggle/working/safeguardk3_export"
os.makedirs(export_dir, exist_ok=True)

# Salin best.pt
if os.path.exists(best_weight):
    shutil.copy(best_weight, os.path.join(export_dir, "best.pt"))
    print("✅ best.pt disalin ke folder export")

# Salin grafik hasil training jika ada
train_plots = glob.glob("/kaggle/working/safeguardk3_training/yolov8s_k3_model/*.png")
for p in train_plots:
    shutil.copy(p, os.path.join(export_dir, os.path.basename(p)))

# Buat arsip zip
zip_output = "/kaggle/working/safeguardk3_best_model"
shutil.make_archive(zip_output, 'zip', export_dir)

print(f"\n🎉 ARSIP MODEL SIAP DIDOWNLOAD: {zip_output}.zip")
print(f"Ukuran: {os.path.getsize(zip_output + '.zip') / (1024 * 1024):.2f} MB")
print("\nLangkah selanjutnya:")
print("1. Download file 'safeguardk3_best_model.zip' dari panel Output Kaggle.")
print("2. Ekstrak dan pindahkan 'best.pt' ke: d:\\Klasifikasi-APD-K3\\artifacts\\best.pt")
print("3. Restart backend FastAPI lokal.")
